In [1]:
%load_ext watermark


In [2]:
import os
import subprocess

os.environ["POLARS_FORCE_NEW_STREAMING"] = "1"

import pandas as pd
import polars as pl
from tqdm import tqdm

from pylib._seed_global_rngs import seed_global_rngs


Covasim 3.1.6 (2024-01-28) — © 2020-2024 by IDM


In [3]:
pd.options.display.float_format = "{:,.1f}".format


In [4]:
%watermark -diwmuv -iv


Last updated: 2025-08-18T19:27:44.048561+00:00

Python implementation: CPython
Python version       : 3.10.12
IPython version      : 7.31.1

Compiler    : GCC 11.4.0
OS          : Linux
Release     : 6.8.0-1031-azure
Machine     : x86_64
Processor   : x86_64
CPU cores   : 4
Architecture: 64bit

polars: 1.29.0
pandas: 2.2.3

Watermark: 2.4.3



In [5]:
teeplot_subdir = "2025-05-30-compscreen-mutcount"
teeplot_subdir


'2025-05-30-compscreen-mutcount'

In [6]:
seed_global_rngs(1)


## Get Data


In [7]:
data_sources = {
    "uk": "https://osf.io/mkjy5/download",
    "multistrain": "https://osf.io/ywmpt/download",
    "vanilla": "https://osf.io/r8skg/download",
    "vanilla-big": "https://osf.io/j4795/download",
    "vanilla-big-1.3x": "https://osf.io/cnp5z/download",
}
tmp_path = f"/tmp/{teeplot_subdir}.pqt"


In [8]:
results = []

for source_name, url in data_sources.items():
    print(f"Downloading {source_name} data from {url}")

    subprocess.run(
        [
            "wget",
            "--tries=5",
            "--show-progress",
            "--progress=bar:force",
            "-O",
            str(tmp_path),
            url,
        ],
        check=True,
    )
    print("done!")

    df = pl.scan_parquet(
        tmp_path,
        low_memory=True,
        retries=5,
    )

    unique_groups = (
        df.unique(
            [
                "trt_name",
                "trt_n_downsample",
                "trt_hsurf_bits",
                "replicate_uuid",
            ]
        )
        .select(
            pl.col("trt_name"),
            pl.col("trt_n_downsample"),
            pl.col("trt_hsurf_bits"),
            pl.col("replicate_uuid"),
        )
        .drop_nans()
        .drop_nulls()
        .collect(engine="streaming")
    )

    for (trt_name, trt_n_downsample, trt_hsurf_bits, replicate_uuid) in tqdm(
        [*unique_groups.iter_rows()],
    ):
        group = df.filter(
            (pl.col("trt_name") == trt_name)
            & (pl.col("trt_n_downsample") == trt_n_downsample)
            & (pl.col("trt_hsurf_bits") == trt_hsurf_bits)
            & (pl.col("replicate_uuid") == replicate_uuid)
        ).collect(engine="streaming")

        group_df = group.to_pandas()
        res = [
            {
                "sum leaf count": group_df.loc[
                    group_df["is_focal_defmut"],
                    "num_leaves",
                ].sum(),
                "sum defmut": group_df["is_focal_defmut"].astype(bool).sum(),
            },
        ]
        results.extend(
            {
                "source_name": source_name,
                "trt_name": trt_name,
                "trt_n_downsample": trt_n_downsample,
                "trt_hsurf_bits": trt_hsurf_bits,
                "replicate_uuid": replicate_uuid,
                **record,
            }
            for record in res
        )


--2025-08-18 19:27:44--  https://osf.io/mkjy5/download
Resolving osf.io (osf.io)... 35.190.84.173
Connecting to osf.io (osf.io)|35.190.84.173|:443... connected.
HTTP request sent, awaiting response... 302 FOUND
Location: https://files.osf.io/v1/resources/37fv8/providers/osfstorage/682c8b8ca9789d8b628f3f38?action=download&direct&version=1 [following]
--2025-08-18 19:27:44--  https://files.osf.io/v1/resources/37fv8/providers/osfstorage/682c8b8ca9789d8b628f3f38?action=download&direct&version=1
Resolving files.osf.io (files.osf.io)... 35.186.214.196
Connecting to files.osf.io (files.osf.io)|35.186.214.196|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 201771388 (192M) [application/octet-stream]
Saving to: ‘/tmp/2025-05-30-compscreen-mutcount.pqt’

/tmp/2025-05-30-com 100%[===================>] 192.42M  88.7MB/s    in 2.2s    

2025-08-18 19:27:47 (88.7 MB/s) - ‘/tmp/2025-05-30-compscreen-mutcount.pqt’ saved [201771388/201771388]



done!


100%|██████████| 70/70 [00:28<00:00,  2.47it/s]
--2025-08-18 19:28:16--  https://osf.io/ywmpt/download
Resolving osf.io (osf.io)... 35.190.84.173
Connecting to osf.io (osf.io)|35.190.84.173|:443... connected.
HTTP request sent, awaiting response... 

302 FOUND
Location: https://files.osf.io/v1/resources/37fv8/providers/osfstorage/682cae3abb3f815770b06cc8?action=download&direct&version=2 [following]
--2025-08-18 19:28:16--  https://files.osf.io/v1/resources/37fv8/providers/osfstorage/682cae3abb3f815770b06cc8?action=download&direct&version=2
Resolving files.osf.io (files.osf.io)... 35.186.214.196
Connecting to files.osf.io (files.osf.io)|35.186.214.196|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1164969026 (1.1G) [application/octet-stream]
Saving to: ‘/tmp/2025-05-30-compscreen-mutcount.pqt’

/tmp/2025-05-30-com 100%[===================>]   1.08G  61.6MB/s    in 78s     

2025-08-18 19:29:37 (14.3 MB/s) - ‘/tmp/2025-05-30-compscreen-mutcount.pqt’ saved [1164969026/1164969026]



done!


...

done!


100%|██████████| 18/18 [00:11<00:00,  1.57it/s]
--2025-08-18 19:32:03--  https://osf.io/cnp5z/download
Resolving osf.io (osf.io)... 

35.190.84.173
Connecting to osf.io (osf.io)|35.190.84.173|:443... connected.
HTTP request sent, awaiting response... 302 FOUND
Location: https://files.osf.io/v1/resources/37fv8/providers/osfstorage/682caa9fbb3f815770b06c6b?action=download&direct&version=1 [following]
--2025-08-18 19:32:03--  https://files.osf.io/v1/resources/37fv8/providers/osfstorage/682caa9fbb3f815770b06c6b?action=download&direct&version=1
Resolving files.osf.io (files.osf.io)... 35.186.214.196
Connecting to files.osf.io (files.osf.io)|35.186.214.196|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 801307036 (764M) [application/octet-stream]
Saving to: ‘/tmp/2025-05-30-compscreen-mutcount.pqt’

/tmp/2025-05-30-com 100%[===================>] 764.19M  81.3MB/s    in 10s     

2025-08-18 19:32:15 (75.0 MB/s) - ‘/tmp/2025-05-30-compscreen-mutcount.pqt’ saved [801307036/801307036]



done!


100%|██████████| 20/20 [00:19<00:00,  1.02it/s]


In [9]:
results_df = pd.DataFrame(results)
results_df.to_csv(
    f"{teeplot_subdir}-raw.csv",
    index=False,
)
results_df


,source_name,trt_name,trt_n_downsample,trt_hsurf_bits,replicate_uuid,sum leaf count,sum defmut
0,uk,Sben2x/Gneu,1000000,0,29143eaa-000f-832e-95e4-edb14e6958b1,9508,600
1,uk,Sben1.3x/Gneu,1000000,64,46ac4a02-4fca-8d5f-b9e6-2ac4c32d1eef,16,16
2,uk,Sben2x/Gneu,1000000,0,c6f2de32-367f-8382-9348-4cbb1101a565,12809,489
3,uk,Sneu/Gneu,1000000,0,d80807db-dd30-8cea-9929-08a538d25d1e,0,0
4,uk,Sben2x/Gneu,1000000,64,f99d06e0-1b1f-8bd4-9755-bd90e4015a04,370,370
...,...,...,...,...,...,...,...
172,vanilla-big-1.3x,Sben1.3x/Gdel1.3x,2000000,0,cb422b5d-9a74-87e8-9654-9bc12d44b453,3765,1270
173,vanilla-big-1.3x,Sben1.3x/Gneu,2000000,0,488c314a-7dbc-88f3-aa36-beada5f1bb05,79730,1807
174,vanilla-big-1.3x,Sben1.3x/Gneu,200000,0,488c314a-7dbc-88f3-aa36-beada5f1bb05,7866,516
175,vanilla-big-1.3x,Sben1.3x/Gneu,200000,0,5a8d2488-6801-88f0-99cd-89a6fda9b042,5437,509


In [10]:
summary_df = results_df.groupby(
    ["source_name", "trt_name", "trt_n_downsample", "trt_hsurf_bits"]
).agg(
    {
        "sum leaf count": ["mean", "std"],
        "sum defmut": ["mean", "std"],
    },
)
summary_df.to_csv(
    f"{teeplot_subdir}-summary.csv",
    index=True,
)
summary_df


sum leaf count  \
                                                                             mean   
source_name      trt_name          trt_n_downsample trt_hsurf_bits                  
multistrain      Sben1.1x/Gdel1.1x 1000000          0                       865.8   
                 Sben1.1x/Gneu     1000000          0                       323.4   
                 Sben1.3x/Gdel1.3x 1000000          0                     1,543.8   
                 Sben1.3x/Gneu     1000000          0                     6,386.8   
                 Sben2x/Gdel2x     1000000          0                    12,510.8   
                 Sben2x/Gneu       1000000          0                   105,072.2   
                 Sneu/Gneu         1000000          0                        92.0   
uk               Sben1.1x/Gdel1.1x 1000000          0                         1.6   
                                                    64                        1.2   
                 Sben1.1x/Gneu     1000000          0                         3.2   
                                                    64                        1.0   
                 Sben1.3x/Gdel1.3x 1000000          0                        67.8   
                                                    64                        9.0   
                 Sben1.3x/Gneu     1000000          0                       130.0   
                                                    64                       10.4   
                 Sben2x/Gdel2x     1000000          0                       559.2   
                                                    64                      210.2   
                 Sben2x/Gneu       1000000          0                     6,436.6   
                                                    64                      348.6   
                 Sneu/Gneu         1000000          0                         1.2   
                                                    64                        0.6   
vanilla          Sben1.1x/Gdel1.1x 1000000          0                       141.2   
                 Sben1.1x/Gneu     1000000          0                       525.0   
                 Sben1.3x/Gdel1.3x 1000000          0                     6,154.4   
                 Sben1.3x/Gneu     1000000          0                    29,039.6   
                 Sben2x/Gdel2x     1000000          0                    18,099.4   
                 Sben2x/Gneu       1000000          0                   299,979.0   
                 Sneu/Gneu         1000000          0                        85.0   
vanilla-big      Sben1.1x/Gdel1.1x 200000           0                       201.0   
                                                    64                       12.7   
                                   2000000          0                     2,684.7   
                 Sben1.1x/Gneu     200000           0                       356.0   
                                                    64                       30.0   
                                   2000000          0                     4,005.7   
vanilla-big-1.3x Sben1.3x/Gdel1.3x 200000           0                       409.2   
                                   2000000          0                     4,892.6   
                 Sben1.3x/Gneu     200000           0                     5,819.0   
                                   2000000          0                    58,494.4   

                                                                             \
                                                                        std   
source_name      trt_name          trt_n_downsample trt_hsurf_bits            
multistrain      Sben1.1x/Gdel1.1x 1000000          0                 438.4   
                 Sben1.1x/Gneu     1000000          0                 241.1   
                 Sben1.3x/Gdel1.3x 1000000          0                 380.3   
                 Sben1.3x/Gneu     1000000          0               2,054.2   
                 Sben2x/Gdel